In [1]:
from datetime import datetime, timedelta
import requests
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 0d24314a-3eda-4b7f-a1db-4afdeae44f7c, 3, Finished, Available, Finished, False)

In [2]:
# Client registry with API credentials and active data sources.

CLIENTES = {
    "<client_slug>": {
        "nombre": "<Client Display Name>",
        "industria": "<industry>",
        "fuentes": {
            "meta": {
                "activo": True,
                "user_token": "<META_USER_TOKEN>",
                "page_token": "<META_PAGE_TOKEN>",
                "page_id": "<META_PAGE_ID>",
                "ad_account_id": "<META_AD_ACCOUNT_ID>"
            },
            "shopify": {
                "activo": True,
                "store_url": "<SHOPIFY_STORE_URL>",
                "access_token": "<SHOPIFY_ACCESS_TOKEN>"
            },
            "google_ads": {"activo": False}
        }
    }
}

StatementMeta(, 0d24314a-3eda-4b7f-a1db-4afdeae44f7c, 4, Finished, Available, Finished, False)

In [3]:
# Ingestion functions — fetch raw data from each platform API and store it in the bronze layer.

def ingest_meta_organic_with_insights(config):
    """
    Fetches all posts from a Meta page and enriches each one with
    lifetime insights (reach, likes, love, wow, comments) via the Graph API.
    Returns a list of enriched post dictionaries.
    """
    token = config["page_token"]
    page_id = config["page_id"]
    
    url = "https://graph.facebook.com/v21.0/" + page_id + "/posts"
    params = {
        "fields": "id,message,created_time,permalink_url,shares",
        "access_token": token,
        "limit": 100
    }
    all_posts = []
    while url:
        r = requests.get(url, params=params)
        if r.status_code != 200:
            break
        data = r.json()
        all_posts.extend(data.get("data", []))
        url = data.get("paging", {}).get("next", None)
        params = None
    
    enriched = []
    for post in all_posts:
        post_id = post["id"]
        r2 = requests.get("https://graph.facebook.com/v21.0/" + post_id + "/insights", params={
            "metric": "post_impressions_unique,post_reactions_like_total,post_reactions_love_total,post_reactions_wow_total,post_activity_by_action_type",
            "access_token": token
        })
        
        post_data = {
            "id": post_id,
            "message": post.get("message", ""),
            "created_time": post.get("created_time", ""),
            "permalink_url": post.get("permalink_url", ""),
            "shares": post.get("shares", {}).get("count", 0),
            "reach": 0,
            "likes": 0,
            "love": 0,
            "wow": 0,
            "comments": 0
        }
        
        if r2.status_code == 200:
            for metric in r2.json().get("data", []):
                name = metric.get("name", "")
                period = metric.get("period", "")
                if period != "lifetime":
                    continue
                value = metric.get("values", [{}])[0].get("value", 0)
                if name == "post_impressions_unique":
                    post_data["reach"] = int(value)
                elif name == "post_reactions_like_total":
                    post_data["likes"] = int(value)
                elif name == "post_reactions_love_total":
                    post_data["love"] = int(value)
                elif name == "post_reactions_wow_total":
                    post_data["wow"] = int(value)
                elif name == "post_activity_by_action_type":
                    if isinstance(value, dict):
                        post_data["comments"] = int(value.get("comment", 0))
        
        print("Post: reach=" + str(post_data["reach"]) + " likes=" + str(post_data["likes"]) + " comments=" + str(post_data["comments"]))
        enriched.append(post_data)
    
    print("Posts con insights: " + str(len(enriched)))
    return enriched


def ingest_meta_ads(config, days_back=30):
    """
    Fetches ad-level insights from Meta Ads API for a given date range.
    Returns a list of insight records with spend, clicks, impressions and actions.
    """
    token = config["user_token"]
    account = config["ad_account_id"]
    date_from = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")
    date_to = datetime.now().strftime("%Y-%m-%d")
    url = "https://graph.facebook.com/v21.0/" + account + "/insights"
    params = {
        "fields": "campaign_name,adset_name,ad_name,impressions,reach,clicks,ctr,cpc,cpm,spend,actions",
        "level": "ad",
        "time_range": json.dumps({"since": date_from, "until": date_to}),
        "time_increment": 1,
        "access_token": token,
        "limit": 500
    }
    all_insights = []
    while url:
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print("Error meta ads: " + str(r.status_code) + " " + r.text[:200])
            break
        data = r.json()
        all_insights.extend(data.get("data", []))
        url = data.get("paging", {}).get("next", None)
        params = None
    print("Ad insights: " + str(len(all_insights)))
    return all_insights


def ingest_shopify(config, resource="products"):
    """
    Fetches a single resource (products, orders or customers) from the Shopify Admin API.
    Returns a list of records for the requested resource.
    """
    store = config["store_url"]
    token = config["access_token"]
    url = "https://" + store + "/admin/api/2026-01/" + resource + ".json"
    headers = {"X-Shopify-Access-Token": token}
    
    r = requests.get(url, headers=headers, params={"limit": 250})
    if r.status_code != 200:
        print("Error shopify: " + str(r.status_code))
        return []
    data = r.json()
    items = data.get(resource, [])
    print("Shopify " + resource + ": " + str(len(items)))
    return items

# Additional ingestion functions for other platforms go here.


def save_bronze(data, client_id, source, resource):
    """
    Saves raw ingested data as a JSON file in the bronze layer of the lakehouse.
    Wraps the data with metadata (source, client, timestamp, record count).
    """
    ts = str(datetime.now())
    date_str = datetime.now().strftime("%Y-%m-%d")
    record = [{
        "source": source,
        "resource": resource,
        "client": client_id,
        "ingestion_timestamp": ts,
        "ingestion_date": date_str,
        "record_count": len(data),
        "raw_data": json.dumps(data)
    }]
    df = spark.createDataFrame(record)
    path = "Files/" + client_id + "/bronze/" + source + "/" + resource
    df.write.mode("append").json(path)
    print("[Bronze] " + client_id + "/" + source + "/" + resource + " -> " + str(len(data)))

StatementMeta(, 0d24314a-3eda-4b7f-a1db-4afdeae44f7c, 5, Finished, Available, Finished, False)

In [4]:
def run_pipeline_completo():
    """
    Runs the full ingestion pipeline for all active clients.
    Iterates over each client in CLIENTES, checks which sources are active,
    and ingests data into the bronze layer. Errors are caught per source
    so a failure in one platform does not stop the rest.
    """
    print("=" * 60)
    print("INICIO PIPELINE - " + str(datetime.now()))
    print("=" * 60)
    
    for client_id, config in CLIENTES.items():
        print("\n" + "=" * 40)
        print("Cliente: " + config["nombre"])
        print("=" * 40)
        
        fuentes = config["fuentes"]
        
        # META ORGANICO + ADS
        if fuentes.get("meta", {}).get("activo"):
            try:
                posts = ingest_meta_organic_with_insights(fuentes["meta"])
                save_bronze(posts, client_id, "meta", "organic_posts_enriched")
            except Exception as e:
                print("[ERROR] Meta organic: " + str(e))
            
            try:
                ads = ingest_meta_ads(fuentes["meta"])
                save_bronze(ads, client_id, "meta", "ad_insights")
            except Exception as e:
                print("[ERROR] Meta ads: " + str(e))
        else:
            print("[SKIP] Meta no activo")
        
        # SHOPIFY
        if fuentes.get("shopify", {}).get("activo"):
            for resource in ["products", "orders", "customers"]:
                try:
                    data = ingest_shopify(fuentes["shopify"], resource)
                    save_bronze(data, client_id, "shopify", resource)
                except Exception as e:
                    print("[ERROR] Shopify " + resource + ": " + str(e))
        else:
            print("[SKIP] Shopify no activo")
        
        # GOOGLE ADS
        if fuentes.get("google_ads", {}).get("activo"):
            print("[TODO] Google Ads - implementar cuando sea necesario")
        else:
            print("[SKIP] Google Ads no activo")
        
        print("[OK] " + config["nombre"] + " completo")
    
    print("\n" + "=" * 60)
    print("PIPELINE FINALIZADO - " + str(datetime.now()))
    print("=" * 60)


# Entry point — execute the full pipeline.
run_pipeline_completo()

StatementMeta(, 0d24314a-3eda-4b7f-a1db-4afdeae44f7c, 6, Finished, Available, Finished, False)

INICIO PIPELINE - 2026-04-13 11:50:48.465341

Cliente: Tuco Home
Post: reach=137 likes=2 comments=2
Post: reach=178 likes=3 comments=0
Post: reach=185 likes=5 comments=3
Post: reach=139 likes=2 comments=0
Post: reach=221 likes=2 comments=0
Post: reach=132 likes=2 comments=0
Post: reach=96 likes=1 comments=0
Post: reach=109 likes=1 comments=0
Post: reach=113 likes=1 comments=0
Post: reach=31486 likes=4 comments=0
Post: reach=124 likes=1 comments=0
Post: reach=3560 likes=15 comments=0
Post: reach=97 likes=2 comments=0
Post: reach=75 likes=1 comments=0
Posts con insights: 14
[Bronze] tuco_home/meta/organic_posts_enriched -> 14
Ad insights: 760
[Bronze] tuco_home/meta/ad_insights -> 760
Shopify products: 3
[Bronze] tuco_home/shopify/products -> 3
Shopify orders: 6
[Bronze] tuco_home/shopify/orders -> 6
Shopify customers: 3
[Bronze] tuco_home/shopify/customers -> 3
[SKIP] Google Ads no activo
[OK] Tuco Home completo

PIPELINE FINALIZADO - 2026-04-13 11:51:10.856214
